# 🧠 AI Concept Explainer
### Explains *any* topic in the simplest, friendliest way possible
> Built for school students, beginners, and non-technical people.

---
**Sections:**
1. 🔧 Setup & Configuration
2. 📖 Simple Concept Explainer
3. 🔁 Analogy Generator
4. 🪜 Step-by-Step Breakdown
5. 🎚️ Difficulty Level Adjuster
6. 🎮 Interactive Q&A Interface

## 🔧 Section 1 — Setup & Configuration
Install dependencies and set your API key here.

In [ ]:
# Install required libraries (run once)
# %pip install openai ipywidgets

import os
import textwrap

# ── OpenAI setup ──────────────────────────────────────────────
# Option A: paste your key directly (dev only, never commit)
# os.environ["OPENAI_API_KEY"] = "sk-..."

# Option B: read from environment (recommended)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

# Global settings
MODEL          = "gpt-4o-mini"   # cheap + fast; swap for "gpt-4o" for better quality
MAX_TOKENS     = 400             # ~half a page — keeps answers concise
TEMPERATURE    = 0.6             # friendly but focused tone
WRAP_WIDTH     = 90              # terminal display width

# Fallback: use Ollama locally (no API key needed)
USE_OLLAMA     = not bool(OPENAI_API_KEY)   # auto-switch when no key
OLLAMA_MODEL   = "llama3"
OLLAMA_URL     = "http://localhost:11434/api/generate"

print("✅ Config loaded.")
print(f"   Backend  : {'Ollama (local)' if USE_OLLAMA else 'OpenAI'}")
print(f"   Model    : {OLLAMA_MODEL if USE_OLLAMA else MODEL}")
print(f"   Max tokens: {MAX_TOKENS}")

In [ ]:
import json, urllib.request

def _call_llm(system_prompt: str, user_prompt: str) -> str:
    """Single helper — works with OpenAI OR local Ollama."""

    if USE_OLLAMA:
        # ── Ollama (local, free) ──────────────────────────────
        payload = json.dumps({
            "model": OLLAMA_MODEL,
            "prompt": f"{system_prompt}\n\n{user_prompt}",
            "stream": False,
            "options": {"temperature": TEMPERATURE, "num_predict": MAX_TOKENS}
        }).encode()
        req = urllib.request.Request(OLLAMA_URL, data=payload,
                                     headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=60) as r:
            return json.loads(r.read())["response"].strip()

    else:
        # ── OpenAI ────────────────────────────────────────────
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        resp = client.chat.completions.create(
            model=MODEL,
            max_tokens=MAX_TOKENS,
            temperature=TEMPERATURE,
            messages=[
                {"role": "system",  "content": system_prompt},
                {"role": "user",    "content": user_prompt},
            ]
        )
        return resp.choices[0].message.content.strip()

print("✅ LLM helper ready.")

## 📖 Section 2 — Simple Concept Explainer
Returns a fully structured, beginner-friendly explanation for any concept.

In [ ]:
EXPLAINER_SYSTEM = """
You are a friendly teacher explaining to a 12-year-old.
Rules:
- Use very simple, short sentences.
- Avoid jargon; if a technical word is needed, explain it instantly.
- Use bullet points and real-life examples.
- Keep the total response under 300 words.
- Always follow this exact structure:

1️⃣ What It Is
(2–3 simple sentences)

2️⃣ Why It Matters
(1–2 sentences, real-life impact)

3️⃣ Simple Example
(one relatable, everyday example)

5️⃣ Quick Summary
(one sentence recap)
""".strip()

def explain_concept(concept: str) -> str:
    """Return a structured beginner-friendly explanation."""
    reply = _call_llm(EXPLAINER_SYSTEM, f"Explain: {concept}")
    return reply

# ── Quick test ────────────────────────────────────────────────
result = explain_concept("the internet")
print(textwrap.fill(result, width=WRAP_WIDTH))

## 🔁 Section 3 — Analogy Generator
Compares any abstract concept to something a 12-year-old already knows.

In [ ]:
ANALOGY_SYSTEM = """
You are a creative teacher. Given a concept, produce ONE short analogy that:
- Compares it to an everyday object or situation (food, school, sports, toys).
- A 12-year-old will instantly recognise.
- Is 2–4 sentences at most.
- Ends with: "That's exactly how [concept] works!"
No jargon. No long paragraphs.
""".strip()

def generate_analogy(concept: str) -> str:
    """Return a relatable real-life analogy for the concept."""
    return _call_llm(ANALOGY_SYSTEM, f"Give an analogy for: {concept}")

# ── Quick test ────────────────────────────────────────────────
analogy = generate_analogy("encryption")
print("🔁 Analogy:")
print(textwrap.fill(analogy, width=WRAP_WIDTH))

## 🪜 Section 4 — Step-by-Step Breakdown Engine
Auto-detects if a concept is a *process* and breaks it into numbered beginner steps.

In [ ]:
# Process keywords — triggers step-by-step mode automatically
PROCESS_KEYWORDS = [
    "how does", "how do", "how to", "process", "steps", "works",
    "explain how", "what happens when", "lifecycle", "flow", "algorithm"
]

STEPS_SYSTEM = """
You are a patient teacher. Break the process into numbered steps.
Rules:
- Each step = 1 short sentence (max 15 words).
- Use simple words a 12-year-old understands.
- Max 7 steps.
- Add one bullet point of detail under each step if needed.
- End with: "✅ Done! That's the whole process."
""".strip()

def is_process_question(question: str) -> bool:
    q = question.lower()
    return any(kw in q for kw in PROCESS_KEYWORDS)

def stepwise_breakdown(concept: str) -> str:
    """Returns numbered steps if concept is a process; else falls back to explain_concept."""
    if is_process_question(concept):
        return _call_llm(STEPS_SYSTEM, f"Break into steps: {concept}")
    else:
        return explain_concept(concept)

# ── Quick test ────────────────────────────────────────────────
process_q = "how does a website load?"
print(f"🔍 Is process? {is_process_question(process_q)}\n")
print(stepwise_breakdown(process_q))

## 🎚️ Section 5 — Difficulty Level Adjuster
Splits the answer into **Basic Idea** and **Advanced Detail** for complex topics.

In [ ]:
# Advanced concept signal words
ADVANCED_KEYWORDS = [
    "quantum", "neural network", "derivative", "recursion", "entropy",
    "eigenvalue", "cryptocurrency", "relativity", "compiler", "operating system",
    "machine learning", "deep learning", "transformer", "fourier", "integral",
    "blockchain", "hashing", "asymptotic", "gradient descent", "backpropagation"
]

LEVELED_SYSTEM = """
You are a patient teacher. Split the answer into two clearly labeled sections:

🟢 Basic Idea  (for a 12-year-old — 3–4 simple sentences, analogy required)
🔵 Advanced Detail  (slightly deeper — still plain English, max 4 sentences)

Keep total response under 300 words.
""".strip()

def is_advanced(concept: str) -> bool:
    c = concept.lower()
    return any(kw in c for kw in ADVANCED_KEYWORDS)

def leveled_explain(concept: str) -> str:
    """
    Simple topics  → standard explain_concept().
    Advanced topics → two-level answer: Basic + Advanced.
    """
    if is_advanced(concept):
        print(f"⚡ Advanced topic detected — showing two levels.\n")
        return _call_llm(LEVELED_SYSTEM, f"Explain: {concept}")
    return explain_concept(concept)

# ── Quick test ────────────────────────────────────────────────
output = leveled_explain("quantum computing")
print(output)

## 🎮 Section 6 — Interactive Q&A Interface
Type any concept below and get an instant, beginner-friendly explanation.
The engine auto-selects: **step-by-step** for processes · **two-level** for advanced topics · **standard** for everything else.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

# ── UI Components ─────────────────────────────────────────────
title_label = widgets.HTML(
    "<h3 style='color:#4A90D9;margin-bottom:4px'>🧠 AI Concept Explainer</h3>"
    "<p style='color:#888;font-size:13px'>Ask anything — in plain English.</p>"
)

input_box = widgets.Text(
    placeholder="e.g. 'blockchain', 'how does WiFi work?', 'quantum computing'",
    layout=widgets.Layout(width="70%"),
    style={"description_width": "0px"}
)

mode_toggle = widgets.ToggleButtons(
    options=["🤖 Auto", "📖 Explain", "🔁 Analogy", "🪜 Steps"],
    description="Mode:",
    style={"description_width": "60px", "button_width": "110px"}
)

ask_btn  = widgets.Button(description="✨ Explain",
                          button_style="primary",
                          layout=widgets.Layout(width="140px", height="36px"))

clear_btn = widgets.Button(description="🗑 Clear",
                           button_style="warning",
                           layout=widgets.Layout(width="100px", height="36px"))

output_area = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #ddd", padding="16px",
        min_height="120px", border_radius="8px",
        width="90%", margin_top="12px"
    )
)

# ── Button handlers ───────────────────────────────────────────
def on_ask(_):
    concept = input_box.value.strip()
    if not concept:
        return

    ask_btn.disabled = True
    ask_btn.description = "⏳ Thinking…"
    output_area.clear_output()

    with output_area:
        print("🔍 Processing…")
        try:
            mode = mode_toggle.value

            if mode == "📖 Explain":
                response = explain_concept(concept)
            elif mode == "🔁 Analogy":
                response = generate_analogy(concept)
            elif mode == "🪜 Steps":
                response = _call_llm(STEPS_SYSTEM, f"Break into steps: {concept}")
            else:
                # Auto: smartest path
                if is_process_question(concept):
                    response = stepwise_breakdown(concept)
                else:
                    response = leveled_explain(concept)

            clear_output(wait=True)
            display(Markdown(f"### 💡 {concept}\n\n---\n\n{response}"))

        except Exception as e:
            clear_output(wait=True)
            print(f"❌ Error: {e}")
            print("👉 Check your API key / Ollama is running.")

    ask_btn.disabled    = False
    ask_btn.description = "✨ Explain"

def on_clear(_):
    output_area.clear_output()
    input_box.value = ""

ask_btn.on_click(on_ask)
clear_btn.on_click(on_clear)

# ── Trigger on Enter key ──────────────────────────────────────
def on_submit(w):
    on_ask(None)

input_box.on_submit(on_submit)

# ── Layout ────────────────────────────────────────────────────
ui = widgets.VBox([
    title_label,
    widgets.HBox([input_box, ask_btn, clear_btn],
                 layout=widgets.Layout(gap="8px", align_items="center")),
    mode_toggle,
    output_area,
], layout=widgets.Layout(padding="16px", gap="10px"))

display(ui)